# MS COCO Images With Most Ground-Truth Objects

This notebook checks which images in the local MS COCO dataset have the most ground-truth object annotations. It is useful when selecting crowded/object-rich examples for XAI experiments.

Dataset root used here:

`/Users/rashid/data/DS/MS_COCO`


In [ ]:
from pathlib import Path
from collections import Counter
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from matplotlib.patches import Rectangle

COCO_ROOT = Path('/Users/rashid/data/DS/MS_COCO')
SPLIT = 'val2017'  # change to 'train2017' if needed

IMAGE_DIR = COCO_ROOT / SPLIT
ANNOTATION_FILE = COCO_ROOT / 'annotations' / 'annotations_trainval2017' / 'annotations' / f'instances_{SPLIT}.json'

print('Image dir:', IMAGE_DIR, IMAGE_DIR.exists())
print('Annotation file:', ANNOTATION_FILE, ANNOTATION_FILE.exists())

## Load COCO Annotations

The count below uses instance annotations from `instances_val2017.json` by default. Set `COUNT_CROWD = False` to ignore `iscrowd=1` annotations.

In [ ]:
COUNT_CROWD = False
MIN_AREA = 0  # set e.g. 100 to ignore tiny annotations

with ANNOTATION_FILE.open('r') as f:
    coco = json.load(f)

images_by_id = {img['id']: img for img in coco['images']}
categories_by_id = {cat['id']: cat['name'] for cat in coco['categories']}

annotations = []
for ann in coco['annotations']:
    if not COUNT_CROWD and ann.get('iscrowd', 0):
        continue
    if ann.get('area', 0) < MIN_AREA:
        continue
    annotations.append(ann)

print('Images:', len(images_by_id))
print('Categories:', len(categories_by_id))
print('Annotations used:', len(annotations))

## Count Objects Per Image

In [ ]:
rows = []
anns_by_image = {}

for ann in annotations:
    anns_by_image.setdefault(ann['image_id'], []).append(ann)

for image_id, img in images_by_id.items():
    anns = anns_by_image.get(image_id, [])
    cat_names = [categories_by_id[ann['category_id']] for ann in anns]
    cat_counts = Counter(cat_names)
    rows.append({
        'image_id': image_id,
        'file_name': img['file_name'],
        'width': img['width'],
        'height': img['height'],
        'n_objects': len(anns),
        'n_categories': len(cat_counts),
        'top_categories': ', '.join(f'{name}:{count}' for name, count in cat_counts.most_common(8)),
        'image_path': str(IMAGE_DIR / img['file_name']),
    })

object_count_df = pd.DataFrame(rows).sort_values(
    ['n_objects', 'n_categories'], ascending=[False, False]
).reset_index(drop=True)

object_count_df.head(20)

## Summary Statistics

In [ ]:
display(object_count_df[['n_objects', 'n_categories', 'width', 'height']].describe())

plt.figure(figsize=(7, 4))
object_count_df['n_objects'].hist(bins=40)
plt.title(f'MS COCO {SPLIT}: ground-truth object count per image')
plt.xlabel('Number of object annotations')
plt.ylabel('Number of images')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Save Ranked CSV

In [ ]:
OUTPUT_DIR = Path('results/coco_gt_object_counts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / f'coco_{SPLIT}_gt_object_counts.csv'
object_count_df.to_csv(csv_path, index=False)
print('Saved:', csv_path.resolve())

## Visualize Images With Most Objects

Set `TOP_N` to control how many crowded images to inspect. Bounding boxes are drawn from ground-truth annotations.

In [ ]:
TOP_N = 12
DRAW_BOXES = True
MAX_BOX_LABELS = 40

def draw_coco_image(ax, row, draw_boxes=True):
    image_id = int(row['image_id'])
    image_path = Path(row['image_path'])
    image = Image.open(image_path).convert('RGB')
    ax.imshow(image)
    ax.set_title(
        f"{image_id:012d}\nobjects={row['n_objects']}, categories={row['n_categories']}",
        fontsize=10,
    )
    ax.axis('off')

    if not draw_boxes:
        return

    anns = anns_by_image.get(image_id, [])
    for i, ann in enumerate(anns):
        x, y, w, h = ann['bbox']
        ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.2, edgecolor='red', alpha=0.8))
        if i < MAX_BOX_LABELS:
            label = categories_by_id[ann['category_id']]
            ax.text(x, max(0, y - 2), label, color='white', fontsize=7,
                    bbox=dict(facecolor='red', alpha=0.55, edgecolor='none', pad=1))

top_df = object_count_df.head(TOP_N)
ncols = 3
nrows = (len(top_df) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]

for ax, (_, row) in zip(axes, top_df.iterrows()):
    draw_coco_image(ax, row, draw_boxes=DRAW_BOXES)

for ax in axes[len(top_df):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Inspect One Image In Detail

In [ ]:
rank = 0  # 0 means the image with the most objects
row = object_count_df.iloc[rank]
image_id = int(row['image_id'])

print(row[['image_id', 'file_name', 'n_objects', 'n_categories', 'top_categories', 'image_path']])

ann_rows = []
for ann in anns_by_image.get(image_id, []):
    ann_rows.append({
        'category': categories_by_id[ann['category_id']],
        'area': ann.get('area'),
        'bbox': ann.get('bbox'),
        'iscrowd': ann.get('iscrowd', 0),
    })

ann_df = pd.DataFrame(ann_rows).sort_values('area', ascending=False)
display(ann_df.head(50))

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
draw_coco_image(ax, row, draw_boxes=True)
plt.show()